<img src='../images/xebia-logo.png' width='300px' align='right' style="padding: 15px">

# Day 3 Morning: Review Session

This notebook brings together everything from the past two days — packaging, code quality, testing, logging, and OOP — into a single hands-on session.

**Don't worry about Polars syntax** — every cell that needs it has the Polars code already written. Your job is to apply the Python patterns around it.

---

### Topics covered

| # | Topic | From day |
|---|---|---|
| 1 | Polars Orientation | Framing the problem |
| 2 | Code quality & refactoring | Day 1 — notebook 02 |
| 3 | Unit tests & fixtures | Day 1 — notebooks 04, 05 |
| 4 | Logging | Day 2 — notebook 06 |
| 5 | Context managers | Day 2 — OOP track |
| 6 | Classes, inheritance & dunders | Day 2 — OOP track |
| 7 | Putting it all together | Today |

## Setup

Run this cell first.

In [ ]:
import logging
import time
import tempfile
import os
from contextlib import contextmanager
from functools import wraps
from pathlib import Path

import polars as pl

print("All imports OK  ✓")
print(f"Polars version: {pl.__version__}")

---

## Part 1 — Polars orientation (10 min)

Run the cells below to get familiar with Polars. No exercises here — just read and run.

In [ ]:
# Creating a DataFrame — like pd.DataFrame(...)
df = pl.DataFrame({
    "name":   ["Luna", "Biscuit", "Rex", "Shadow"],
    "type":   ["Cat",  "Dog",     "Dog", "Cat"],
    "age_yr": [2,      5,         1,     3],
})
df

In [ ]:
# .filter() — like df[df['col'] == value]
df.filter(pl.col("type") == "Dog")

In [ ]:
# .with_columns() — add or replace a column
# Polars DataFrames are immutable: this returns a NEW DataFrame, it does not change df
df.with_columns(
    (pl.col("age_yr") * 365).alias("age_days")
)

In [ ]:
# .select() — choose columns (like df[['col1', 'col2']])
df.select(["name", "type"])

In [ ]:
# Chaining — this is the idiomatic Polars style
result = (
    df
    .filter(pl.col("age_yr") >= 2)
    .with_columns((pl.col("age_yr") * 365).alias("age_days"))
    .select(["name", "type", "age_days"])
)
result

In [ ]:
# .pipe() — pass a DataFrame through a function, exactly like pandas
def add_senior_flag(df: pl.DataFrame) -> pl.DataFrame:
    """Mark animals older than 7 years as senior."""
    return df.with_columns(
        (pl.col("age_yr") > 7).cast(pl.Int8).alias("is_senior")
    )

df.pipe(add_senior_flag)

---

## Part 2 — Refactoring

Below is a working but messy pipeline. Read it carefully to understand what it does, then refactor it into clean, well-named functions in the exercise below.

In [ ]:
# ── The messy version ─────────────────────────────────────────────────────────
# Read this, understand what it does, then refactor it below.

def _build_dataset_messy(df: pl.DataFrame) -> pl.DataFrame:

    ## Cleaning ##
    df = df.rename({"Age upon Outcome": "age_upon_outcome", "Sex Upon Outcome": "sex_upon_outcome"})
    df = df.with_columns(pl.col("age_upon_outcome").str.strip_chars())
    df = df.drop_nulls(subset=["animal_type", "sex_upon_outcome"])

    ## Features ##
    df = df.with_columns(
        (pl.col("animal_type").str.to_lowercase() == "dog")
        .cast(pl.Int8).alias("is_dog")
    )
    df = df.with_columns(
        (
            pl.col("name").is_null() |
            pl.col("name").str.to_lowercase().is_in({"unknown", "", "n/a", "none"})
        ).not_().alias("has_name")
    )
    df = df.with_columns(
        pl.when(pl.col("sex_upon_outcome").str.to_lowercase().str.ends_with("female"))
        .then(pl.lit("female"))
        .when(pl.col("sex_upon_outcome").str.to_lowercase().str.ends_with("male"))
        .then(pl.lit("male"))
        .otherwise(pl.lit("unknown"))
        .alias("sex")
    )
    df = df.with_columns(
        pl.when(pl.col("sex_upon_outcome").str.to_lowercase().str.contains("neutered|spayed"))
        .then(pl.lit("fixed"))
        .when(pl.col("sex_upon_outcome").str.to_lowercase().str.contains("intact"))
        .then(pl.lit("intact"))
        .otherwise(pl.lit("unknown"))
        .alias("neutered")
    )
    df = df.with_columns(
        pl.when(pl.col("age_upon_outcome").str.contains("year"))
        .then(pl.col("age_upon_outcome").str.extract(r"(\d+)").cast(pl.Int32) * 365)
        .when(pl.col("age_upon_outcome").str.contains("month"))
        .then(pl.col("age_upon_outcome").str.extract(r"(\d+)").cast(pl.Int32) * 30)
        .when(pl.col("age_upon_outcome").str.contains("week"))
        .then(pl.col("age_upon_outcome").str.extract(r"(\d+)").cast(pl.Int32) * 7)
        .otherwise(pl.lit(1))
        .alias("age_days")
    )

    ## Filter ##
    df = df.filter(
        (pl.col("is_dog") == 1) &
        pl.col("has_name") &
        (pl.col("age_days") < 365 * 10)
    )
    return df

### <mark>Exercise 2a: Split into helper functions</mark>

Refactor the messy version above into:
- Named constants for magic values
- Small helper functions returning `pl.Expr`
- Three pipeline stage functions: `clean_data`, `add_features`, `filter_adoptable`
- A public entry point `build_dataset` that uses `.pipe()`

In [ ]:
# ── Constants ─────────────────────────────────────────────────────────────────
# TODO: define named constants for all magic values
# ("dog" for _check_is_dog and the set of unknown name values for _check_has_name)


# ── Feature expressions ───────────────────────────────────────────────────────
# TODO: implement each helper as a function returning pl.Expr

def _check_is_dog(col: str = "animal_type") -> pl.Expr:
    ...

def _check_has_name(col: str = "name", unknown_values: frozenset = ...) -> pl.Expr:
    ...

def _get_sex(col: str = "sex_upon_outcome") -> pl.Expr:
    # HINT: female must be checked before male
    ...

def _get_neutered(col: str = "sex_upon_outcome") -> pl.Expr:
    ...

def _compute_age_days(col: str = "age_upon_outcome") -> pl.Expr:
    ...


# ── Pipeline stages ───────────────────────────────────────────────────────────

def clean_data(df: pl.DataFrame) -> pl.DataFrame:
    """Drop nulls, fix types, rename messy columns."""
    ...


def add_features(df: pl.DataFrame) -> pl.DataFrame:
    """Add all engineered features in a single pass."""
    # HINT: pass all expressions to a single .with_columns() call
    ...


def filter_adoptable(df: pl.DataFrame) -> pl.DataFrame:
    """Keep only animals likely to be adoptable."""
    ...


# ── Public entry point ────────────────────────────────────────────────────────

def build_dataset(df: pl.DataFrame) -> pl.DataFrame:
    # TODO: chain the three stages together using .pipe()
    ...

In [ ]:
sample = pl.DataFrame({
    "animal_type":      ["Dog",          "Cat",           "Dog"],
    "name":             ["Biscuit",       "unknown",       "Rex"],
    "Sex Upon Outcome": ["Neutered Male", "Intact Female", "Spayed Female"],
    "Age upon Outcome": ["2 years",       "6 months",      "3 weeks"],
    "outcome_type":     ["Adoption",      "Transfer",      "Return to Owner"],
})

result = build_dataset(sample)
result

**Expected output — check yours matches:**

| animal_type | name | sex_upon_outcome | age_upon_outcome | outcome_type | is_dog | has_name | sex | neutered | age_days |
|---|---|---|---|---|---|---|---|---|---|
| str | str | str | str | str | i8 | bool | str | str | i32 |
| "Dog" | "Biscuit" | "Neutered Male" | "2 years" | "Adoption" | 1 | true | "male" | "fixed" | 730 |
| "Dog" | "Rex" | "Spayed Female" | "3 weeks" | "Return to Owner" | 1 | true | "female" | "fixed" | 21 |

---

## Part 3 — Unit tests (30 min)

### <mark>Exercise 3a: Write unit tests for your helper functions</mark>

Write parametrized tests for each helper function.

Remember:
- One test function per helper, with `@pytest.mark.parametrize` for different cases
- Test the **happy path**, **case insensitivity**, **edge cases** (null, fallthrough, zero)
- Keep dtype tests as separate standalone functions

In [ ]:
import pytest
import ipytest
ipytest.autoconfig()


# ── _check_is_dog ─────────────────────────────────────────────────────────────

@pytest.mark.parametrize("animal_type, expected", [
    # TODO: add test cases
    # ([...], [...]),
])
def test_check_is_dog(animal_type, expected):
    df = pl.DataFrame({"animal_type": animal_type})
    assert df.select(_check_is_dog())["is_dog"].to_list() == expected

def test_check_is_dog_dtype():
    df = pl.DataFrame({"animal_type": ["Dog"]})
    # TODO: assert the dtype is pl.Int8
    ...


# ── _check_has_name ───────────────────────────────────────────────────────────

@pytest.mark.parametrize("name, expected", [
    # TODO: add test cases
    # Cover: happy path, placeholders, case insensitive, null
])
def test_check_has_name(name, expected):
    df = pl.DataFrame({"name": name})
    assert df.select(_check_has_name())["has_name"].to_list() == expected

def test_check_has_name_custom_unknowns():
    # TODO: pass a custom unknown_values frozenset containing "pending"
    # and assert "pending" returns False but "Biscuit" returns True
    ...


# ── _get_sex ──────────────────────────────────────────────────────────────────

@pytest.mark.parametrize("sex_upon_outcome, expected", [
    # TODO: add test cases
    # HINT: include a case that checks "female" is matched before "male"
])
def test_get_sex(sex_upon_outcome, expected):
    df = pl.DataFrame({"sex_upon_outcome": sex_upon_outcome})
    assert df.select(_get_sex())["sex"].to_list() == expected


# ── _get_neutered ─────────────────────────────────────────────────────────────

@pytest.mark.parametrize("sex_upon_outcome, expected", [
    # TODO: add test cases
    # Cover: neutered → fixed, spayed → fixed, intact, fallthrough
])
def test_get_neutered(sex_upon_outcome, expected):
    df = pl.DataFrame({"sex_upon_outcome": sex_upon_outcome})
    assert df.select(_get_neutered())["neutered"].to_list() == expected


# ── _compute_age_days ─────────────────────────────────────────────────────────

@pytest.mark.parametrize("age_upon_outcome, expected", [
    # TODO: add test cases
    # Cover: years, months, weeks, zero, unparseable fallback
])
def test_compute_age_days(age_upon_outcome, expected):
    df = pl.DataFrame({"age_upon_outcome": age_upon_outcome})
    assert df.select(_compute_age_days())["age_days"].to_list() == expected

def test_compute_age_days_dtype():
    df = pl.DataFrame({"age_upon_outcome": ["2 years"]})
    # TODO: assert the dtype is pl.Int32
    ...

In [ ]:
ipytest.run()

---

## Part 4 — Logging (20 min)

### <mark>Exercise 4a: Implement `setup_logger`</mark>

Implement the `setup_logger` function below. It should:
- Use logger name `"animal_shelter"`
- Clear any existing handlers before adding new ones (prevents duplicate output on re-run)
- Log to stdout via a `StreamHandler`
- Format: `"%(asctime)s - %(name)s - %(levelname)s - %(message)s"`
- Apply `level` to both the logger and the handler

In [ ]:
import logging


def setup_logger(level: int = logging.INFO) -> None:
    """Set up the animal_shelter logger."""
    # TODO: get the "animal_shelter" logger
    # TODO: clear existing handlers
    # TODO: set the logger level
    # TODO: create a StreamHandler, set its level, attach a Formatter
    # TODO: attach the handler to the logger
    ...


# ── Test level filtering ──────────────────────────────────────────────────────
# At DEBUG level — both lines should appear
setup_logger(level=logging.DEBUG)
logger = logging.getLogger("animal_shelter")
logger.debug("this is a debug message")
logger.info("this is an info message")

print()

# At INFO level — only the info line should appear
setup_logger(level=logging.INFO)
logger = logging.getLogger("animal_shelter")
logger.debug("you should NOT see this")
logger.info("you should see this")

### <mark>Exercise 4b: Add logging to `add_features`</mark>

Modify `add_features` to log:
- At `INFO` level: how many rows are being processed
- At `DEBUG` level: the column names of the input DataFrame
- At `INFO` level: how many columns were added when done

In [ ]:
setup_logger(level=logging.DEBUG)
logger = logging.getLogger("animal_shelter")


def add_features(df: pl.DataFrame) -> pl.DataFrame:
    """Add all engineered features in a single pass."""
    # TODO: log at INFO — how many rows
    # TODO: log at DEBUG — input column names
    result = df.with_columns(
        _check_is_dog(),
        _check_has_name(),
        _get_sex(),
        _get_neutered(),
        _compute_age_days(),
    )
    # TODO: log at INFO — feature engineering complete, N columns added
    return result


build_dataset(sample)

### <mark>Exercise 4c: Write a `log_step` decorator</mark>

Write a decorator that, when applied to a pipeline function:
1. Logs the function name and input row count at `INFO` level
2. Runs the function
3. Logs the output row count, and how many rows were dropped (if any)

In [ ]:
def log_step(func):
    """Decorator that logs input/output row counts for a pipeline step."""
    @wraps(func)
    def wrapper(df: pl.DataFrame, *args, **kwargs) -> pl.DataFrame:
        # TODO: log function name and input row count
        result = func(df, *args, **kwargs)
        # TODO: log output row count, and rows dropped if any
        return result
    return wrapper


@log_step
def remove_unknown_outcomes(df: pl.DataFrame) -> pl.DataFrame:
    """Drop rows where outcome_type is unknown."""
    return df.filter(pl.col("outcome_type") != "Unknown")


@log_step
def remove_missing_ages(df: pl.DataFrame) -> pl.DataFrame:
    """Drop rows where age is missing or 'Unknown'."""
    return df.filter(pl.col("age_upon_outcome") != "Unknown")


test_df = pl.DataFrame({
    "animal_type":      ["Dog",           "Cat",           "Dog",           "Cat"],
    "name":             ["Rex",           "unknown",       "Biscuit",       "Luna"],
    "Sex Upon Outcome": ["Neutered Male",  "Intact Female", "Spayed Female", "Unknown"],
    "Age upon Outcome": ["2 years",        "Unknown",       "6 months",      "1 year"],
    "outcome_type":     ["Adoption",       "Transfer",      "Unknown",       "Return to Owner"],
})

cleaned = (
    test_df
    .pipe(clean_data)
    .pipe(remove_unknown_outcomes)
    .pipe(remove_missing_ages)
)
# Expected log: 4 rows → 3 rows (1 dropped) → 2 rows (1 dropped)
cleaned

---

## Part 5 — Context managers (20 min)

### <mark>Exercise 5a: A `timer` context manager</mark>

Write a context manager that measures how long a block takes and logs the result.
Use `time.perf_counter()` and the `@contextmanager` decorator.

In [ ]:
@contextmanager
def timer(description: str = "block"):
    """Context manager that logs the execution time of the enclosed block.

    Usage:
        with timer("preprocessing"):
            result = add_features(df)
    """
    # TODO: record start time
    try:
        yield
    finally:
        # TODO: compute elapsed time and log it
        ...


with timer("feature engineering"):
    result = build_dataset(sample)

# Expected:
# INFO - feature engineering took 0.0012s

### <mark>Exercise 5b: A `dataframe_checkpoint` context manager</mark>

Write a context manager that:
1. On entry: saves a DataFrame to a temp CSV file and logs where it saved it
2. Yields the path to that file
3. On exit: deletes the file and logs that it was cleaned up

In [ ]:
@contextmanager
def dataframe_checkpoint(df: pl.DataFrame, name: str = "checkpoint"):
    """Save a DataFrame snapshot to a temp file, yield the path, then clean up.

    Usage:
        with dataframe_checkpoint(my_df, "after_cleaning") as path:
            print(f"Inspect at: {path}")
            # file exists here
        # file is deleted here
    """
    # TODO: create a NamedTemporaryFile with suffix ".csv" and prefix f"{name}_"
    # TODO: write the DataFrame to the temp file
    # TODO: log where the checkpoint was saved and how many rows
    # TODO: yield the path
    # TODO: in the finally block, delete the file and log that it was cleaned up
    ...


with dataframe_checkpoint(sample, "sample_data") as path:
    print(f"File exists during context: {Path(path).exists()}")
    reloaded = pl.read_csv(path)
    print(f"Rows in checkpoint: {reloaded.height}")

print(f"File exists after context: {Path(path).exists()}")

---

## Part 6 — Classes, inheritance & dunders (25 min)

### <mark>Exercise 6a: Build a `DataPipeline` class</mark>

Build a class that wraps a list of pipeline steps and runs them in order.

Implement:
- `__init__(self, steps)` — store the list of step functions
- `__len__` — return the number of steps
- `__repr__` — return something like `DataPipeline(steps=['clean_data', 'add_features'])`
- `run(self, df)` — apply each step in order, log each step name, return the final DataFrame

In [ ]:
class DataPipeline:
    """A simple sequential pipeline of DataFrame transformation steps."""

    def __init__(self, steps: list):
        # TODO
        ...

    def __len__(self) -> int:
        # TODO
        ...

    def __repr__(self) -> str:
        # TODO
        ...

    def run(self, df: pl.DataFrame) -> pl.DataFrame:
        """Run all steps in order and return the result."""
        # TODO
        ...


pipeline = DataPipeline(steps=[
    remove_unknown_outcomes,
    remove_missing_ages,
    add_features,
])

print(pipeline)      # should show step names
print(len(pipeline)) # should be 3
output = pipeline.run(test_df)
output

### <mark>Exercise 6b: A child class — `TimedPipeline`</mark>

Create a child class `TimedPipeline` that inherits from `DataPipeline` and overrides `run()` to:
- Record how long each step takes
- Print a timing summary table after all steps complete

In [ ]:
class TimedPipeline(DataPipeline):
    """A DataPipeline that records and reports the time taken by each step."""

    def run(self, df: pl.DataFrame) -> pl.DataFrame:
        """Run all steps and print a timing summary."""
        # TODO: run each step, record timing, print summary table
        ...


timed = TimedPipeline(steps=[
    clean_data,
    remove_unknown_outcomes,
    remove_missing_ages,
])

output = timed.run(test_df)
output

---

## Part 7 — Putting it all together (20 min)

### <mark>Exercise 7: Extend `DataPipeline` with iteration and context manager support</mark>

Add two more capabilities to `DataPipeline`:

**Part A** — Make it iterable with `__iter__` and `__next__`:
```python
for step in pipeline:
    print(step.__name__)
```

**Part B** — Make it a context manager with `__enter__` and `__exit__`:
```python
with DataPipeline(steps=[...]) as pipeline:
    result = pipeline.run(df)
# On exit: log that the pipeline finished (or failed)
```

**Note:** `__exit__` receives `(exc_type, exc_val, exc_tb)`. Return `False` to let exceptions propagate normally.

In [ ]:
class DataPipeline:
    """A sequential DataFrame pipeline with logging, iteration, and context manager support."""

    def __init__(self, steps: list):
        self._steps = steps
        self._current = 0  # needed for __next__

    def __len__(self) -> int:
        return len(self._steps)

    def __repr__(self) -> str:
        names = [s.__name__ for s in self._steps]
        return f"DataPipeline(steps={names})"

    def __iter__(self):
        # TODO: reset _current and return self
        ...

    def __next__(self):
        # TODO: return next step, raise StopIteration when exhausted
        ...

    def __enter__(self):
        # TODO: log that the pipeline is starting
        ...

    def __exit__(self, exc_type, exc_val, exc_tb):
        # TODO: log success or failure, return False
        ...

    def run(self, df: pl.DataFrame) -> pl.DataFrame:
        """Run all steps in order."""
        for step in self:  # uses __iter__ and __next__
            logger.info("Running step: %s", step.__name__)
            df = step(df)
        return df


# Part A — iteration
pipeline = DataPipeline(steps=[clean_data, remove_unknown_outcomes, remove_missing_ages])

print("Steps in the pipeline:")
for step in pipeline:
    print(" -", step.__name__)

print()

# Part B — context manager
with DataPipeline(steps=[clean_data, remove_unknown_outcomes, remove_missing_ages]) as p:
    result = p.run(test_df)

result

---

## Session complete

| Pattern | Where you used it |
|---|---|
| Single-responsibility functions | `_check_is_dog`, `_get_sex`, etc. in Part 2 |
| Unit tests + parametrize | Part 3 |
| `logging` + decorators | `log_step` in Part 4 |
| `@contextmanager` with `try/finally` | `timer`, `dataframe_checkpoint` in Part 5 |
| Class with `__len__`, `__repr__` | `DataPipeline` in Part 6 |
| Inheritance | `TimedPipeline(DataPipeline)` in Part 6 |
| `__iter__` / `__next__` | Part 7a |
| `__enter__` / `__exit__` | Part 7b |